# Stage 4: Hyperparameter Tuning with Optuna

This notebook performs hyperparameter optimization for the top models (XGBoost and LightGBM) on the **full dataset** for each segment. It uses Optuna for efficient searching, a SQLite backend for checkpointing, and CUDA for acceleration.

## 1. Setup and Imports

In [6]:
import gc
import os
import warnings

import joblib
import lightgbm as lgb
import numpy as np

# Optuna for Hyperparameter Tuning
import optuna
import pandas as pd

# Scikit-Learn and Models
import xgboost as xgb

# TQDM for progress bars
from tqdm.notebook import tqdm

# --- Configuration ---
warnings.filterwarnings("ignore")
SEGMENTS_DIR = "data/segments/"
MODELS_DIR = "models/"
RESULTS_FILE = "tuning_results.csv"
DB_URL = "sqlite:///tuning.db"  # Database for checkpointing
N_TRIALS = 50  # Number of tuning trials per model/segment

os.makedirs(MODELS_DIR, exist_ok=True)

## 2. Utility Functions (Memory, Metrics, Features)

In [2]:
def reduce_mem_usage(df, verbose=True):
    start_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage of dataframe is {start_mem:.2f} MB")
    for col in df.columns:
        col_type = df[col].dtype
        if (
            col_type != object
            and col_type.name != "category"
            and "datetime" not in col_type.name
        ):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if (
                    c_min > np.finfo(np.float32).min
                    and c_max < np.finfo(np.float32).max
                ):
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage after optimization is: {end_mem:.2f} MB")
    return df


def calculate_wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100


def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


def get_features_and_target(df):
    TARGET_COL = "BaseLoad"
    cols_to_drop = [
        TARGET_COL,
        "TradeDate",
        "TradeTime",
        "LoadProfile",
        "RateGroup",
        "Submission",
        "LossAdjustedLoad",
        "LoadBL",
        "LoadLAL",
        "GenBL",
        "GenLAL",
        "Solar_Status",
        "Created",
    ]
    features = [col for col in df.columns if col not in cols_to_drop]
    return df[features], df[TARGET_COL]

## 3. Optuna Objective Function

This is the core of the tuning process. For each trial, Optuna suggests a set of hyperparameters. This function trains a model using those parameters on a training split and evaluates it on a validation split. The resulting error score (WAPE or MAPE) is returned to Optuna, which it then tries to minimize.

In [3]:
def objective(trial, model_name, X_train, y_train, X_val, y_val, is_solar):
    """The objective function for Optuna to minimize."""
    if model_name == "LightGBM":
        params = {
            "objective": "regression_l1",
            "metric": "mae",
            "n_estimators": 1000,
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
            "num_leaves": trial.suggest_int("num_leaves", 20, 300),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.7, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.7, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "verbose": -1,
            "n_jobs": -1,
            "seed": 42,
            "device": "gpu",  # CUDA Acceleration
        }
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )

    elif model_name == "XGBoost":
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "n_estimators": 1000,
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0, 0.5),
            "seed": 42,
            "tree_method": "gpu_hist",  # CUDA Acceleration
            "early_stopping_rounds": 100,
        }
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    preds = model.predict(X_val)

    # Return the primary metric for Optuna to minimize
    if is_solar:
        score = calculate_wape(y_val, preds)
    else:
        score = calculate_mape(y_val, preds)

    return score

## 4. Main Execution Pipeline

This is the main orchestrator. It loops through each segment and each of the two selected models. For each pair, it:
1.  Loads the **full dataset**.
2.  Creates or loads an Optuna study from the `tuning.db` file (this is the checkpoint system).
3.  Runs the optimization process, showing a progress bar.
4.  Saves the best model artifact to the `/models` directory.
5.  Appends the trial results to a master list.

In [4]:
models_to_tune = ["LightGBM", "XGBoost"]
segment_files = [f for f in os.listdir(SEGMENTS_DIR) if f.endswith(".csv")]

all_trials_list = []

# Outer progress bar for all jobs
pbar = tqdm(total=len(segment_files) * len(models_to_tune), desc="Overall Progress")

for segment_file in segment_files:
    segment_name = segment_file.replace(".csv", "")
    print(f"\n{'=' * 60}\nProcessing Segment: {segment_name}\n{'=' * 60}")

    df = pd.read_csv(
        os.path.join(SEGMENTS_DIR, segment_file),
        index_col="Timestamp",
        parse_dates=True,
    )
    df = reduce_mem_usage(df)

    is_solar = "Solar" in segment_file
    X, y = get_features_and_target(df)

    # Create a single train/validation split for the objective function
    # Using the last 3 months as a validation set for tuning
    split_date = df.index.max() - pd.DateOffset(months=3)
    train_mask = df.index <= split_date
    val_mask = df.index > split_date
    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]

    for model_name in models_to_tune:
        pbar.set_description(f"Tuning {model_name} on {segment_name}")
        study_name = f"{segment_name}_{model_name}"

        # Create study with RDB storage for checkpointing
        # load_if_exists=True allows resuming from interruptions
        study = optuna.create_study(
            direction="minimize",
            study_name=study_name,
            storage=DB_URL,
            load_if_exists=True,
        )

        # Run optimization
        study.optimize(
            lambda trial: objective(
                trial, model_name, X_train, y_train, X_val, y_val, is_solar
            ),
            n_trials=N_TRIALS,
            show_progress_bar=True,
        )

        print(
            f"Finished tuning for {study_name}. Best trial score: {study.best_value:.4f}"
        )

        # Store results
        trial_df = study.trials_dataframe()
        trial_df["model"] = model_name
        trial_df["segment"] = segment_name
        all_trials_list.append(trial_df)

        # Retrain best model on full dataset and save it
        print("Retraining best model on the full dataset...")
        best_params = study.best_params
        if model_name == "LightGBM":
            final_model = lgb.LGBMRegressor(
                device="gpu",
                objective="regression_l1",
                metric="mae",
                n_estimators=1000,
                **best_params,
            )
            final_model.fit(
                X,
                y,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(100, verbose=False)],
            )
        elif model_name == "XGBoost":
            final_model = xgb.XGBRegressor(
                tree_method="gpu_hist",
                objective="reg:squarederror",
                eval_metric="rmse",
                n_estimators=1000,
                **best_params,
            )
            final_model.fit(X, y, eval_set=[(X_val, y_val)], verbose=False)

        # Save the trained model artifact
        model_filename = os.path.join(
            MODELS_DIR, f"{segment_name}_{model_name}_best.joblib"
        )
        joblib.dump(final_model, model_filename)
        print(f"Best model saved to {model_filename}")

        pbar.update(1)
        del final_model
        gc.collect()

    del df, X, y, X_train, y_train, X_val, y_val
    gc.collect()

pbar.close()

# Save all trial results to a single CSV file
if all_trials_list:
    full_results_df = pd.concat(all_trials_list, ignore_index=True)
    full_results_df.to_csv(RESULTS_FILE, index=False)
    print(f"\nAll tuning trial results saved to '{RESULTS_FILE}'.")

Overall Progress:   0%|          | 0/12 [00:00<?, ?it/s]


Processing Segment: Medium_Scale_Industries_Non-Solar
Memory usage of dataframe is 220.28 MB
Memory usage after optimization is: 111.06 MB


[I 2025-09-28 00:02:28,154] Using an existing study with name 'Medium_Scale_Industries_Non-Solar_LightGBM' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-09-28 00:03:01,590] Trial 227 finished with value: 27.443752600480575 and parameters: {'learning_rate': 0.025105917038429813, 'num_leaves': 59, 'max_depth': 12, 'min_child_samples': 53, 'feature_fraction': 0.746432359349467, 'bagging_fraction': 0.7476824889385822, 'bagging_freq': 4}. Best is trial 194 with value: 27.436848449480667.
[I 2025-09-28 00:03:31,206] Trial 228 finished with value: 27.443415807374592 and parameters: {'learning_rate': 0.02495173214439604, 'num_leaves': 60, 'max_depth': 12, 'min_child_samples': 53, 'feature_fraction': 0.7429348420521427, 'bagging_fraction': 0.7464753767802126, 'bagging_freq': 4}. Best is trial 194 with value: 27.436848449480667.
[I 2025-09-28 00:04:07,869] Trial 229 finished with value: 27.444955415667337 and parameters: {'learning_rate': 0.02513654312291187, 'num_leaves': 59, 'max_depth': 12, 'min_child_samples': 53, 'feature_fraction': 0.7457730352481848, 'bagging_fraction': 0.749438242806655, 'bagging_freq': 4}. Best is trial 194 with

...

[I 2025-09-28 04:32:43,424] Trial 0 finished with value: 51.80820565219671 and parameters: {'learning_rate': 0.04257778515232958, 'num_leaves': 267, 'max_depth': 9, 'min_child_samples': 21, 'feature_fraction': 0.803950552206091, 'bagging_fraction': 0.8745640002594031, 'bagging_freq': 5}. Best is trial 0 with value: 51.80820565219671.
[I 2025-09-28 04:33:08,137] Trial 1 finished with value: 51.7851301380407 and parameters: {'learning_rate': 0.0625857140560488, 'num_leaves': 299, 'max_depth': 11, 'min_child_samples': 8, 'feature_fraction': 0.7115362503206548, 'bagging_fraction': 0.7829643792991792, 'bagging_freq': 5}. Best is trial 1 with value: 51.7851301380407.
[I 2025-09-28 04:33:47,351] Trial 2 finished with value: 51.76414739175097 and parameters: {'learning_rate': 0.02551841872142285, 'num_leaves': 171, 'max_depth': 9, 'min_child_samples': 44, 'feature_fraction': 0.7115822937225731, 'bagging_fraction': 0.720301359116297, 'bagging_freq': 2}. Best is trial 2 with value: 51.7641473917

[I 2025-09-28 04:50:51,475] A new study created in RDB with name: Small_Scale_Industries_Solar_XGBoost


Best model saved to models/Small_Scale_Industries_Solar_LightGBM_best.joblib


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-09-28 04:50:55,520] Trial 0 finished with value: 52.046226501464844 and parameters: {'learning_rate': 0.06692768294652872, 'max_depth': 11, 'subsample': 0.7774840113743785, 'colsample_bytree': 0.8284856966820731, 'min_child_weight': 10, 'gamma': 0.3071348503478216}. Best is trial 0 with value: 52.046226501464844.
[I 2025-09-28 04:50:57,614] Trial 1 finished with value: 51.93082046508789 and parameters: {'learning_rate': 0.09831162948447808, 'max_depth': 9, 'subsample': 0.8887211523206316, 'colsample_bytree': 0.7247296104120663, 'min_child_weight': 6, 'gamma': 0.20960678234953806}. Best is trial 1 with value: 51.93082046508789.
[I 2025-09-28 04:51:00,209] Trial 2 finished with value: 51.787757873535156 and parameters: {'learning_rate': 0.05340343679400849, 'max_depth': 8, 'subsample': 0.983217258272181, 'colsample_bytree': 0.701403886363479, 'min_child_weight': 9, 'gamma': 0.43999817948729053}. Best is trial 2 with value: 51.787757873535156.
[I 2025-09-28 04:51:06,570] Trial 3 f

## 5. Final Summary and Analysis

In [7]:
if os.path.exists(RESULTS_FILE):
    print("--- Summary of Best Trials ---")
    results_df = pd.read_csv(RESULTS_FILE)

    # Find the best trial for each study (segment + model)
    best_trials = results_df.loc[
        results_df.groupby(["segment", "model"])["value"].idxmin()
    ]

    # Pivot for easy comparison
    summary_pivot = best_trials.pivot_table(
        index="segment", columns="model", values="value"
    )

    # Add a column to show the overall best model for the segment
    summary_pivot["best_model"] = summary_pivot.idxmin(axis=1)
    summary_pivot["best_score"] = summary_pivot.min(axis=1, numeric_only=True)

    print("Lower scores (WAPE/MAPE) are better.")
    display(summary_pivot[["LightGBM", "XGBoost", "best_model", "best_score"]])
else:
    print("No results file found. Please run the pipeline first.")

--- Summary of Best Trials ---
Lower scores (WAPE/MAPE) are better.


model,LightGBM,XGBoost,best_model,best_score
segment,,,,
Medium_Scale_Industries_Non-Solar,27.436848,27.486458,LightGBM,27.436848
Medium_Scale_Industries_Solar,36.065541,36.076969,LightGBM,36.065541
Residential_Non-Solar,22.255197,22.274977,LightGBM,22.255197
Residential_Solar,41.050292,41.128159,LightGBM,41.050292
Small_Scale_Industries_Non-Solar,28.000480,27.975967,XGBoost,27.975967
Small_Scale_Industries_Solar,51.670176,51.648582,XGBoost,51.648582
